# KSDD2 — Dataset exploration

Goal: understand the data before building the PyTorch `Dataset`.

We will check:
- class balance;
- image sizes;
- how much of each positive image is occupied by the defect;
- a few image/mask examples.


In [ ]:
import csv
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ksdd2"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"


## 1. Load the frozen manifests

Using the CSV files means every experiment sees exactly the same train, validation and test samples.


In [ ]:
def read_manifest(name):
    with (SPLITS_DIR / f"{name}.csv").open() as file:
        return list(csv.DictReader(file))

train_rows = read_manifest("train")
val_rows = read_manifest("val")
test_rows = read_manifest("test")

for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    defective = sum(int(row["label"]) for row in rows)
    print(f"{name}: {len(rows)} samples ({defective} defective, {len(rows) - defective} clean)")


## 2. Class balance

The dataset is strongly imbalanced: most images do not contain a defect. This matters later for metrics, sampling and loss selection.


In [ ]:
split_rows = {"train": train_rows, "val": val_rows, "test": test_rows}

clean_counts = []
defect_counts = []

for rows in split_rows.values():
    defects = sum(int(row["label"]) for row in rows)
    defect_counts.append(defects)
    clean_counts.append(len(rows) - defects)

x = np.arange(len(split_rows))
plt.bar(x, clean_counts, label="clean")
plt.bar(x, defect_counts, bottom=clean_counts, label="defective")
plt.xticks(x, split_rows.keys())
plt.ylabel("samples")
plt.title("Class distribution")
plt.legend()
plt.show()


## 3. Image sizes

KSDD2 images have similar but not identical dimensions. We need to know this before batching tensors in PyTorch.


In [ ]:
sizes = Counter()

for row in train_rows:
    with Image.open(DATA_ROOT / row["image"]) as image:
        sizes[image.size] += 1

widths = [width for (width, height), count in sizes.items() for _ in range(count)]
heights = [height for (width, height), count in sizes.items() for _ in range(count)]

print(f"unique sizes: {len(sizes)}")
print(f"width range: {min(widths)}–{max(widths)} px")
print(f"height range: {min(heights)}–{max(heights)} px")
print("most common sizes:")
for size, count in sizes.most_common(5):
    print(f"  {size}: {count}")


## 4. Defect area

For positive samples, measure the fraction of image pixels marked as defect. Small values mean the segmentation task has strong pixel-level imbalance as well as image-level imbalance.


In [ ]:
defect_areas = []

for row in train_rows:
    if row["label"] == "0":
        continue

    with Image.open(DATA_ROOT / row["mask"]) as mask:
        mask_array = np.asarray(mask)
        defect_fraction = np.count_nonzero(mask_array) / mask_array.size
        defect_areas.append(defect_fraction)

defect_percentages = np.array(defect_areas) * 100

print(f"positive masks: {len(defect_percentages)}")
print(f"median defect area: {np.median(defect_percentages):.3f}%")
print(f"min defect area: {defect_percentages.min():.3f}%")
print(f"max defect area: {defect_percentages.max():.3f}%")

plt.hist(defect_percentages, bins=20)
plt.xlabel("defect area (%)")
plt.ylabel("samples")
plt.title("Defect area in positive training samples")
plt.show()


## 5. Visual inspection

Look at a few positive samples and verify that the mask is aligned with the visible defect.


In [ ]:
positive_rows = [row for row in train_rows if row["label"] == "1"][:3]

fig, axes = plt.subplots(len(positive_rows), 3, figsize=(12, 7))

for row_axes, row in zip(axes, positive_rows):
    with Image.open(DATA_ROOT / row["image"]) as image, Image.open(DATA_ROOT / row["mask"]) as mask:
        row_axes[0].imshow(image)
        row_axes[0].set_title(row["image"])

        row_axes[1].imshow(mask, cmap="gray")
        row_axes[1].set_title("mask")

        row_axes[2].imshow(image)
        row_axes[2].imshow(mask, alpha=0.4, cmap="Reds")
        row_axes[2].set_title("overlay")

    for ax in row_axes:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Questions to answer after running the notebook

- How severe is the image-level class imbalance?
- How small are the defects relative to the full image?
- Do image dimensions vary enough that resizing or padding will be required?
- Do the masks visually align with the defects?
- Are there suspicious or unusual examples that should be inspected separately?

Write the observed answers here after running the notebook. Those observations will guide preprocessing and the first PyTorch baseline.
